# Optional Checkpoint · Gold Model Handoff Check

![Power BI handoff map](../../../assets/images/day1_03_powerbi_handoff_map.png)

How this visual helps: it turns the optional checkpoint into a handoff checklist. The checkpoint is successful only when Gold objects, KPI smoke tests and connection details are ready for Day 2.

This short notebook is an optional checkpoint after Workshop 1. It does not teach the full Power BI semantic model. That official module is `day2_01_powerbi_semantic_model.ipynb`.

Use this checkpoint only to confirm that the W1 Gold objects exist, return sensible KPI values and are ready for the Day 2 Power BI module.

## Business Scenario

Workshop 1 created the governed Gold contract. Before leaving Day 1, the trainer may run a short handoff check to prove that the model is complete enough for Day 2.

The goal is operational readiness, not another Power BI lesson.

## Learning Objectives

By the end of this optional checkpoint you can:

- confirm the five W1 Gold objects exist,
- produce a compact object inventory,
- run a KPI smoke check,
- state what will be covered in the official Day 2 Power BI module.

## Reference Links for the Handoff Check

| Handoff check | Working definition | Official reference |
|---|---|---|
| Object inventory | prove that expected Gold tables/views exist before Power BI handoff | [Information schema](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-information-schema) |
| Table metadata | inspect columns, comments and table-level properties | [DESCRIBE TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-aux-describe-table) |
| Table definition | retrieve the SQL definition of an existing table or view | [SHOW CREATE TABLE](https://docs.databricks.com/aws/en/sql/language-manual/sql-ref-syntax-aux-show-create-table) |
| SQL Warehouse connection | get hostname and HTTP path for Power BI connection | [SQL warehouse connection details](https://docs.databricks.com/aws/en/compute/sql-warehouse/) |
| Metric view readiness | decide whether shared KPI definitions should move closer to Databricks | [Unity Catalog metric views](https://docs.databricks.com/aws/en/business-semantics/metric-views/) |


## Setup

Resolve the participant catalog. `00_setup` executes `USE CATALOG`, so SQL cells can use two-part names such as `gold.fact_sales_dashboard`.

In [0]:
%run ../../../setup/00_setup

### Configuration

Confirm the active catalog and schemas before running the checkpoint.

In [0]:
display(spark.createDataFrame([
    ("CATALOG", CATALOG),
    ("BRONZE", BRONZE),
    ("SILVER", SILVER),
    ("GOLD", GOLD),
], ["Variable", "Value"]))

### Runtime Pre-check

Workshop 1 must have created the complete Gold model. If this fails, run `solution/w1_gold_kpi_solution.ipynb` first.

In [0]:
required_tables = [
    f"{GOLD}.dim_date",
    f"{GOLD}.dim_customer",
    f"{GOLD}.dim_product",
    f"{GOLD}.fact_sales",
    f"{GOLD}.fact_sales_dashboard",
]
missing = [t for t in required_tables if not spark.catalog.tableExists(t)]
if missing:
    for table in missing:
        print("[MISSING]", table)
    raise Exception("Pre-check failed. Complete Workshop 1 before running this optional checkpoint.")
print("[OK] Workshop 1 Gold model is available.")

## 1. Gold Object Inventory

Definition: an object inventory is a simple completeness check. It confirms that the expected tables are present and gives the trainer a row-count baseline.

Expected observation: all five Gold objects should return non-zero row counts.

In [0]:
%sql
SELECT 'gold.dim_date' AS object_name, COUNT(*) AS rows FROM gold.dim_date
UNION ALL
SELECT 'gold.dim_customer', COUNT(*) FROM gold.dim_customer
UNION ALL
SELECT 'gold.dim_product', COUNT(*) FROM gold.dim_product
UNION ALL
SELECT 'gold.fact_sales', COUNT(*) FROM gold.fact_sales
UNION ALL
SELECT 'gold.fact_sales_dashboard', COUNT(*) FROM gold.fact_sales_dashboard
ORDER BY object_name

## 2. KPI Smoke Check

![Gold quality gates](../../../assets/images/gold_quality_gate.png)

How this visual helps: it frames the checkpoint as a quality gate, not just a row-count query. Gold is ready for Power BI only when grain, duplicates, reconciliation, business rules and ownership checks pass.

Definition: a smoke check proves that the model can answer the baseline KPI question without testing every semantic-model choice.

Rule for this course: revenue and margin count completed rows only.


In [0]:
%sql
SELECT
  ROUND(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 2) AS revenue,
  ROUND(SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END), 2) AS gross_margin,
  COUNT(DISTINCT CASE WHEN is_completed THEN order_id END) AS completed_orders,
  ROUND(
    100.0 * SUM(CASE WHEN is_completed THEN line_margin ELSE 0 END)
    / NULLIF(SUM(CASE WHEN is_completed THEN line_revenue ELSE 0 END), 0),
    2
  ) AS margin_rate_pct
FROM gold.fact_sales_dashboard

## 3. Handoff Notes for Day 2

This checkpoint intentionally avoids the full Power BI topic set. Day 2 owns the semantic-model decisions, consumption-mode tradeoffs, report-measure ownership, refresh design and final Power BI handoff packet.

In [0]:
%sql
SELECT 'W1 Gold model' AS handoff_item, 'ready for Day2_01 if all five Gold objects exist' AS expected_status
UNION ALL
SELECT 'KPI baseline', 'ready if smoke check returns non-null revenue and completed_orders'
UNION ALL
SELECT 'Semantic model decisions', 'covered in day2_01_powerbi_semantic_model.ipynb'
UNION ALL
SELECT 'Dataset performance layer', 'built in Workshop 2 after Day2_01' 

## Summary

Use this notebook only as an optional Day 1 checkpoint. The required flow continues with Workshop 1 complete, then the official Power BI semantic-model module in Day 2.